# MIPROv2

Jointly propose instructions and demonstrations, then use validation feedback to search their combinations.

**Use it when:** Both prompt wording and examples may matter and you can spend more compile calls for a joint search.

**What compilation changes:** Instruction plus up to two bootstrapped and two labeled demonstrations.

Important configuration in this benchmark:

- `auto='light'` search budget
- seed 42 with minibatched validation
- Sol proposes prompts; Luna runs the task

Every notebook loads the same 74-row dataset and frozen, pair-grouped
train/validation/test membership before it can compile anything.
The test partition is deliberately baseline-adversarial, so these scores teach
optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'miprov2'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='miprov2'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.MIPROv2(
    metric=exact_match, prompt_model=reflection_lm, task_model=task_lm,
    auto='light', max_bootstrapped_demos=2, max_labeled_demos=2, seed=42,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, valset=valset, minibatch=True,
    requires_permission_to_run=False,
)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 20-example test split. The baseline has its own notebook; all other
notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: miprov2
task model: openai/gpt-5.6-luna
final test accuracy: 95.0% (19/20)
optimization time: 455.5s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/miprov2.json
- canonical prompt: chapter06/results/final_prompts/miprov2.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Inspect instruction and demos together: MIPROv2 optimizes their combination, so attributing the result to either one alone would be misleading.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (1724 characters):
You are the final forensic reviewer in a high-stakes technical publishing audit. A mistaken verdict could either allow undisclosed AI-rewritten documentation into a safety-critical release or falsely accuse a human author, so assess each passage carefully and impartially.

Determine whether the supplied passage is AI-generated or AI-paraphrased (`True`) versus human-written (`False`). Because paired passages may contain nearly identical technical facts, judge the **writing style rather than the topic, correctness, or sophistication of the content**.

Look for clusters of evidence:

- **AI tendencies:** conspicuously polished transitions; elevated, archaic, or needlessly formal diction; excessive syntactic smoothing; balanced or uniformly fluent sentence structures; formal punctuation; conversion of concise steps or lists into flowing prose; awkwardly ornate wording; generic broad claims; mild embellishment or reinterpreta

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.